In [1]:
# ── 셀 1: GPU 확인 ──────────────────────────────────────────
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


CUDA 사용 가능: True
GPU: Tesla T4


In [2]:
# ── 셀 2: 패키지 설치 ────────────────────────────────────────
!pip install -q sentence-transformers FlagEmbedding


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.2 MB/s eta 0:00:00


In [3]:
# ── 셀 3: chunks 업로드 ──────────────────────────────────────
# chunks_ho.json        — 호 단위 원본
# chunks_ho_xref.json   — 호 단위 + cross_refs
# chunks_jo.json        — 조 단위 (Hierarchical RAG fetch용)
import os
for fname in ['chunks_ho.json', 'chunks_ho_xref.json', 'chunks_jo.json']:
    if os.path.exists(fname):
        os.remove(fname)
        print(f'기존 파일 삭제: {fname}')

from google.colab import files
uploaded = files.upload()  # 세 파일 선택


Saving chunks_ho.json to chunks_ho.json
Saving chunks_ho_xref.json to chunks_ho_xref.json
Saving chunks_jo.json to chunks_jo.json


In [4]:
# ── 셀 4: 청크 로드 ──────────────────────────────────────────
import json

def load_chunks(path, exclude_parent=True):
    with open(path, encoding='utf-8') as f:
        chunks = json.load(f)
    seen = {}
    for c in chunks:
        seen[c['chunk_id']] = c
    chunks = list(seen.values())
    if exclude_parent:
        chunks = [c for c in chunks if not c.get('is_parent', False)]
    return chunks

# ho 원본 (child만)
ho_chunks      = load_chunks('chunks_ho.json', exclude_parent=True)
ho_texts       = [c['text']     for c in ho_chunks]
ho_chunk_ids   = [c['chunk_id'] for c in ho_chunks]

# ho xref (child만)
xref_chunks    = load_chunks('chunks_ho_xref.json', exclude_parent=True)
xref_texts     = [c['text']     for c in xref_chunks]
xref_chunk_ids = [c['chunk_id'] for c in xref_chunks]

# jo (전체)
jo_chunks      = load_chunks('chunks_jo.json', exclude_parent=False)
jo_texts       = [c['text']     for c in jo_chunks]
jo_chunk_ids   = [c['chunk_id'] for c in jo_chunks]

print(f'ho 원본 (child):  {len(ho_chunks)}개')
print(f'ho xref (child):  {len(xref_chunks)}개')
print(f'jo 청크:          {len(jo_chunks)}개')


ho 원본 (child):  5463개
ho xref (child):  5463개
jo 청크:          1126개


In [5]:
# ── 셀 5: 평가셋 생성 ────────────────────────────────────────
import random

def make_eval_dataset(chunks, n=200):
    templates = [
        '다음 내용은 어떤 규정을 설명하나요?',
        '이 조항은 무엇에 관한 내용인가요?',
        '다음 규정과 관련된 조문을 찾아주세요.',
        '계약 실무에서 아래 내용은 어떤 법적 근거에 해당하나요?'
    ]
    eval_data = []
    qid = 1
    for chunk in chunks:
        text = chunk['text']
        if len(text) < 80:
            continue
        query = random.choice(templates) + '\n\n' + text[:120]
        eval_data.append({
            'query_id':      f'eval_{qid:03d}',
            'query':         query,
            'relevant_docs': [chunk['chunk_id']]
        })
        qid += 1
    random.shuffle(eval_data)
    return eval_data[:n]

eval_ho   = make_eval_dataset(ho_chunks,   n=200)
eval_xref = make_eval_dataset(xref_chunks, n=200)
eval_jo   = make_eval_dataset(jo_chunks,   n=200)

print(f'ho 평가셋:   {len(eval_ho)}개')
print(f'xref 평가셋: {len(eval_xref)}개')
print(f'jo 평가셋:   {len(eval_jo)}개')


ho 평가셋:   200개
xref 평가셋: 200개
jo 평가셋:   200개


In [6]:
# ── 셀 6: 평가 함수 ──────────────────────────────────────────
import gc
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def evaluate_model(model_name, texts, chunk_ids, eval_data, batch_size=8):
    print(f'\n===== {model_name} =====')
    model = SentenceTransformer(model_name, device='cuda')
    doc_vectors = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True
    )
    recall1 = recall5 = recall10 = mrr = 0
    for item in eval_data:
        gt_docs = set(item['relevant_docs'])
        qvec = model.encode(
            item['query'],
            normalize_embeddings=True,
            convert_to_numpy=True
        )
        scores     = cosine_similarity([qvec], doc_vectors)[0]
        ranked_ids = [chunk_ids[i] for i in np.argsort(scores)[::-1]]
        if any(d in gt_docs for d in ranked_ids[:1]):  recall1  += 1
        if any(d in gt_docs for d in ranked_ids[:5]):  recall5  += 1
        if any(d in gt_docs for d in ranked_ids[:10]): recall10 += 1
        for rank, d in enumerate(ranked_ids, 1):
            if d in gt_docs:
                mrr += 1 / rank
                break
    n = len(eval_data)
    del doc_vectors, model
    gc.collect()
    torch.cuda.empty_cache()
    return {
        'Model':     model_name,
        'Recall@1':  round(recall1  / n, 4),
        'Recall@5':  round(recall5  / n, 4),
        'Recall@10': round(recall10 / n, 4),
        'MRR':       round(mrr      / n, 4),
    }


In [7]:
# ── 셀 7: Part 1 — BGE-M3 성능 비교 (ho / xref / jo) ────────
import pandas as pd

results = []

print('▶ [1/3] ho 원본')
r = evaluate_model('BAAI/bge-m3', ho_texts, ho_chunk_ids, eval_ho, batch_size=8)
r['데이터'] = 'chunks_ho'
results.append(r)

print('\n▶ [2/3] ho xref')
r = evaluate_model('BAAI/bge-m3', xref_texts, xref_chunk_ids, eval_xref, batch_size=8)
r['데이터'] = 'chunks_ho_xref'
results.append(r)

print('\n▶ [3/3] jo')
r = evaluate_model('BAAI/bge-m3', jo_texts, jo_chunk_ids, eval_jo, batch_size=8)
r['데이터'] = 'chunks_jo'
results.append(r)

df = pd.DataFrame(results)[['데이터', 'Model', 'Recall@1', 'Recall@5', 'Recall@10', 'MRR']]
display(df)


▶ [1/3] ho 원본

===== BAAI/bge-m3 =====


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/683 [00:00<?, ?it/s]


▶ [3/3] jo

===== BAAI/bge-m3 =====


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/141 [00:00<?, ?it/s]

,데이터,Model,Recall@1,Recall@5,Recall@10,MRR
0,chunks_ho,BAAI/bge-m3,0.835,0.965,0.995,0.8871
1,chunks_ho_xref,BAAI/bge-m3,0.810,0.960,1.000,0.8762
2,chunks_jo,BAAI/bge-m3,0.980,1.000,1.000,0.9888


In [8]:
# ── 셀 8: Part 2 — BGE-M3 dense + sparse 임베딩 및 저장 ─────
from FlagEmbedding import BGEM3FlagModel
import numpy as np
import json

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

def embed_and_save(texts, chunk_ids, vec_path, sparse_path, label):
    print(f'\n▶ {label} 임베딩 ({len(texts)}개)')
    output = model.encode(
        texts,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False,
        batch_size=8,
    )
    np.savez(
        vec_path,
        vectors=output['dense_vecs'],
        chunk_ids=np.array(chunk_ids)
    )
    sparse = [{k: float(v) for k, v in sw.items()} for sw in output['lexical_weights']]
    with open(sparse_path, 'w', encoding='utf-8') as f:
        json.dump(sparse, f)
    print(f'  ✅ 저장 완료: {vec_path}, {sparse_path}  shape={output["dense_vecs"].shape}')

# [1/3] ho 원본
embed_and_save(ho_texts,   ho_chunk_ids,   'vectors_ho.npz',      'sparse_weights_ho.json',      'ho 원본')

# [2/3] ho xref
embed_and_save(xref_texts, xref_chunk_ids, 'vectors_ho_xref.npz', 'sparse_weights_ho_xref.json', 'ho xref')

# [3/3] jo
embed_and_save(jo_texts,   jo_chunk_ids,   'vectors_jo.npz',      'sparse_weights_jo.json',      'jo')


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


▶ ho 원본 임베딩 (5463개)


Inference Embeddings: 100%|██████████| 683/683 [00:19<00:00, 35.40it/s]


  ✅ 저장 완료: vectors_ho.npz, sparse_weights_ho.json  shape=(5463, 1024)

▶ ho xref 임베딩 (5463개)


Inference Embeddings: 100%|██████████| 683/683 [00:20<00:00, 33.39it/s]


  ✅ 저장 완료: vectors_ho_xref.npz, sparse_weights_ho_xref.json  shape=(5463, 1024)

▶ jo 임베딩 (1126개)


Inference Embeddings: 100%|██████████| 141/141 [00:11<00:00, 12.48it/s]


  ✅ 저장 완료: vectors_jo.npz, sparse_weights_jo.json  shape=(1126, 1024)


In [9]:
# ── 셀 9: 다운로드 ───────────────────────────────────────────
from google.colab import files

for fname in [
    'vectors_ho.npz',      'sparse_weights_ho.json',
    'vectors_ho_xref.npz', 'sparse_weights_ho_xref.json',
    'vectors_jo.npz',      'sparse_weights_jo.json',
]:
    files.download(fname)
    print(f'다운로드: {fname}')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

다운로드: vectors_ho.npz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

다운로드: vectors_jo.npz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

다운로드: sparse_weights_jo.json
